# 07 — Stale documents

**Scenario.** The 2024 classification standard allowed pasting Confidential text into any internal chatbot. That rule is **superseded**. If retrieval prefers it, the assistant gives the wrong control.

1. Search for the withdrawn "Sensitive" label and the old chatbot rule.
2. Compare scores before and after the superseded penalty.

Context:

- RAG systems often index current and historical records for audit.
- Retrieval must know which records are live, superseded, draft, or withdrawn.
- A source-of-truth rule beats semantic similarity when documents disagree.

Edge cases:

- A stale document can use the exact words in the question and rank first.
- Removing old documents can hurt investigations, so this demo keeps them but penalizes them.
- Dates alone are not enough; status metadata is explicit.


Definitions and context:

| Concept | Meaning |
| --- | --- |
| Superseded | Kept for history but no longer the active rule |
| Current | Preferred for live answers |
| Source of truth | System or document family that wins conflicts |
| Recency signal | Score boost or penalty based on age/status |
| Audit retention | Keeping old records for investigations |

Stale documents are dangerous because they often contain exactly the terms a
user asks about. A production pipeline should preserve them for audit but mark
them clearly so live answers prefer current policy.


In [ ]:
import _course
from lib.pipeline import apply_acl, embed, load_index, rerank, rrf

index = load_index()
q = "Can I paste Confidential client data into an internal chatbot? Is Sensitive still a label?"
vec = embed([q])[0]
fused = rrf([index.dense(vec, 12), index.sparse(q, 12)])
fused = apply_acl(fused, "employee")

print("FUSED (no recency penalty)")
for h in fused[:6]:
    print(f"  {h.fused_score:.3f}  {h.chunk.status:11} {h.chunk.doc_id} / {h.chunk.section}")
print()
print("RERANKED (superseded -0.4, current +0.08)")
for h in rerank(q, fused, keep=6):
    print(f"  {h.rerank_score:.3f}  {h.chunk.status:11} {h.chunk.doc_id} / {h.chunk.section}")


Live answers should follow POL-CLASS-2026, not POL-CLASS-2024. Next: `08-access-control.ipynb`.
